# Object oriented decision tree
We have seen already how to write functions to train and use a decision tree classifier. 
This notebook contains an object oriented version of the same functionality. 
It's not quite as simple as copying the functions we had before into the class structure but the similarities remain clear.
The choice of whether to use object oriented programming or not is one of style. 
Think about which you prefer and why. 

In [ ]:
from collections import namedtuple 
from sklearn import datasets
import math

# load the dataset
iris = datasets.load_iris()

We use named tuples to represent the samples and tests. We could use classes but because these objects are so simple it's convenient to use this really lightweight structure instead.
We define a class to hold a set of samples. 
This class will be responsible for generating the tests associated with a set of samples and for applying a test to a set of samples. 
The samples themselves contain only the raw data, the sample set object will also be aware of the meaning of that data i.e. the names of features and target classes. 
We use this knowledge to make tests a little more readable: instead of specifying the index of the property to be tested we can use the name of the property. 

In [ ]:
Sample = namedtuple('Sample', ['properties', 'class_index'])
NumericTest = namedtuple('NumericTest', ['property_name', 'value'])

class SampleSet:
    def __init__(self, samples, property_names, class_names):
        # a list of Sample tuples
        self.samples = samples
        self.property_names = property_names
        self.class_names = class_names
        
    def add_sample(self, sample):
        self.samples.append(sample)
        
    def get_class_name(self, class_index):
        return self.class_names[class_index]
    
    def get_property_name(self, property_index):
        return self.property_names[property_index]
    
    def get_property_index(self, property_name):
        return self.property_names.index(property_name)
    
    def get_numeric_tests(self):
        """
            Define a test for each property of each sample.
            The test contains the property index and the value of this property for this sample.
            the idea is that we will later use these property values 
            to find a value that partitions the set of samples in a 'good' way 

            Returns:
                list of NumericTest tuples (property_name, value)
        """
        tests = set()
        for s in self.samples:
            for property_index, property_value in enumerate(s.properties):
                property_name = self.get_property_name(property_index)
                tests.add(NumericTest(property_name, property_value))
        return list(tests)
    
    def split_by_numeric_test(self, test):
        """
            A test is a property index and value.
            We will test whether the relevant property is greater than the specified value 
            for each sample in samples.

            Parameters:
                test: a NumericTest tuple

            Returns:
                tuple of (leq_samples, gt_samples)
                where leq_samples: set of samples with relevant property not greater than the value given in test
                and gt_samples: set of samples with relevant property greater than the value given in test
        """
        leq_samples = SampleSet([], self.property_names, self.class_names)
        gt_samples = SampleSet([], self.property_names, self.class_names)

        for s in self.samples:
            if s.properties[self.get_property_index(test.property_name)] > test.value:
                gt_samples.add_sample(s)
            else:
                leq_samples.add_sample(s)

        return leq_samples, gt_samples
    
    def classification_frequency(self):
        """
            Return a dictionary {class_name: frequency}
            For each classification, the number of samples in this set
            with that classification.
        """
        freq_dict = {}
        for class_name in self.class_names:
            freq_dict[class_name] = 0

        for s in self.samples:
            class_name = self.get_class_name(s.class_index)
            freq_dict[class_name] += 1
        return freq_dict

The next class just holds the functions related to entropy. Normally we think of a class like a blueprint and expect to make multiple objects from it. In this case, however, it makes more sense to just have one entropy calculator object. Instead of creating a new object by calling `EntropyCalculator()`, we will call `EntropyCalculator.getInstance()`. The class keeps a copy of its one instance which is passed back to the calling code.

In [ ]:
class EntropyCalculator():
    """
        Singleton class, useful methods for calculating entropy
    """
    
    # single class instance
    __instance = None
   
    @staticmethod 
    def getInstance():
        if EntropyCalculator.__instance == None:
            EntropyCalculator()
        return EntropyCalculator.__instance
   
    def __init__(self):
        if EntropyCalculator.__instance != None:
            raise Exception("Use getInstance()")
        else:
            EntropyCalculator.__instance = self

    @staticmethod
    def from_outcome_distribution(number_vector):
        """
            Calculate entropy given a list of the number of outcomes in each category.

            Parameters:
                number_vector list(ints): Numbers of outcomes for different categories.

            Returns:
                A float representing the entropy of an event with distribution as number_vector

            len(number_vector) is the number of different possible outcomes.
            number_vector[i] is the number of events with outcome i.
            sum(number_vector) is the total number of events.
            Therefore number_vector[i]/sum(number_vector) is the probability of outcome i.
        """
        total = sum(number_vector) # N, total number of events 
        entropy = 0 # initialise

        for number in number_vector: # n_i, number of outcomes of type i
            if number == 0:
                continue # outcome did not occur, has no effect on entropy
            proportion = number/total # P_i
            entropy -= proportion * math.log(proportion, 2) # P_i * log_2(P_i)

        return entropy

    @classmethod
    def average_entropy(cls, *distributions):
        """
            Given a list of freq distributions find the average entropy
        """
        weighted_entropy_sum = 0
        total_count = 0
        
        for freq_list in distributions:
            entropy = cls.from_outcome_distribution(freq_list)
            count = sum(freq_list)
            total_count += count
            weighted_entropy_sum += entropy*count
            
        avg_entropy = weighted_entropy_sum/total_count
        return avg_entropy

Now we make a decision tree class. In the base case a decision tree is just a node associated with a prediction. To model this we make a class for a decision tree leaf. 
Some decision tree functions are called recursively so there must be matching functions in the decision tree leaf class and decision tree class. 
Observe that the methods `display()` and `classify()` appear in both, and that the implementation relies on the two classes using the same names for these methods.

The ``learn_tree()`` method is an alternative way of creating a DecisionTree object.
`@classmethod` means that instead of the method being given an object as the first argument (`self`) it is given the _class_ (typically assigned to a parameter called `cls`). Therefore it can access static methods of the class but not instance methods.

In [ ]:
Prediction = namedtuple('Prediction', ['sample', 'predicted_class'])

class DecisionTreeLeaf():
    def __init__(self, class_frequencies):
        self.class_frequencies = class_frequencies
        self.classification = self.decide_class()
        
    def display(self, prefix='', indent=0):
        """
            return a string stating the numbers of training examples for each classification
        """
        output = " "*6*indent + prefix
        classification = ['{} ({})'.format(k, v) for k,v in self.class_frequencies.items() if v != 0]
        
        if len(classification) > 1:
            output += 'Mixed, '
        else:
            output += 'Classified, '
        
        output += ','.join(classification)
        return output
    
    def decide_class(self):
        """
            determine classification of this node
            as most common class of associated training samples
        """
        max_count = 0
        node_class = None
        for name, count in self.class_frequencies.items():
            if count > max_count:
                max_count = count
                node_class = name
        return node_class
    
    def classify(self, sampleSet):
        """
            returns predictions for each sample in sampleSet
            according to the classification of this node
        """
        predictions = []
        for s in sampleSet.samples:
            predictions.append(Prediction(s, self.classification))
        
        return predictions

class DecisionTree():    
    entropyHelper = EntropyCalculator.getInstance()  
    
    def __init__(self, test, left_tree, right_tree):
        # in this example all tests are NumericTest tuples
        self.test = test
        # left and right trees either a DecisionTree or DecisionTreeLeaf
        self.left_tree = left_tree
        self.right_tree = right_tree

    @staticmethod
    def find_test_with_min_entropy(sampleSet):
        """
            get all the tests associated with the sample set
            and select the best among them according to the 
            average entropy after applying the test to the sample set.
        """
        # get all tests associated with this sample set
        tests = sampleSet.get_numeric_tests()
        
        best = None
        for test in tests:
            # apply test to sampleSet, splits into two sample sets
            leq_samples, gt_samples = sampleSet.split_by_numeric_test(test)
            
            # get class freq distribution for both sample sets
            f_leq = leq_samples.classification_frequency().values()
            f_gt = gt_samples.classification_frequency().values()
            
            # calculate average entropy of the two sample sets
            entropy = DecisionTree.entropyHelper.average_entropy(f_leq, f_gt)
            
            # keep track of test with min average entropy
            if best is None or entropy <= best[1]:
                best = (test, entropy)

        return best 
    
    @classmethod
    def learn_tree(cls, sampleSet):
        """
            Learn a decision tree from the provided training data.
            
            Parameters:
                sampleSet: a SampleSet object
                
            Returns:
                a DecisionTree object OR DecisionTreeLeaf object
        """
        class_freq_dict = sampleSet.classification_frequency()
        
        # Check if all samples are in one class. Then we are done.
        if len(class_freq_dict) <= 1:
            return DecisionTreeLeaf(class_freq_dict)

        initial_entropy = cls.entropyHelper.from_outcome_distribution(class_freq_dict.values())
        test, entropy = cls.find_test_with_min_entropy(sampleSet)

        # unable to improve entropy by applying any test
        if entropy == initial_entropy:
            return DecisionTreeLeaf(class_freq_dict)
        
        leq, gt = sampleSet.split_by_numeric_test(test)

        # Make decision trees for samples remaining after each of
        # the test outcomes
        left_tree = cls.learn_tree(leq)
        right_tree = cls.learn_tree(gt)

        return cls(test, left_tree, right_tree)
    
    def display(self, prefix='', indent=0):
        """
            returns a string representation of the tree with indentation
            
            Parameters:
                prefix: string of additional information to include before writing this tree
                indent: amount of white space to add before writing this tree
        """
        output = " "*6*indent + prefix
        
        # print test
        test = self.test
        output += "test: "+ str(test[0]) + " " + str(test[1])
        
        # print left and right trees
        left = self.left_tree.display('leq - ', indent+1)
        right = self.right_tree.display('gt - ', indent+1)
        output += "\n" + left + "\n" + right
        
        return output
            
    def classify(self, sampleSet):
        """
            Classify the given samples using the decision tree.

            Parameters:
                sampleSet: a SampleSet object

            Return:
                a list of Prediction tuples
        """
        # split using the test at the root of this tree
        left_samples, right_samples = sampleSet.split_by_numeric_test(self.test)
        
        # call the child tree to classify the two resulting sample sets
        left_predictions = self.left_tree.classify(left_samples)
        right_predictions = self.right_tree.classify(right_samples)
      
        # combine the left and right prediction lists into a single list
        predictions = left_predictions + right_predictions
        return predictions
    
    # a printable representation of the object
    def __repr__(self):
        return self.display()

Now we can train and test a decision tree using our classes. 

In [ ]:
iris_samples = [Sample(iris.data[i], iris.target[i]) for i in range(len(iris.data))]

test_set = []
training_set = []
class_indices = []

for i, s in enumerate(iris_samples):
    if s.class_index not in class_indices:
        test_set.append(s)
        class_indices.append(s.class_index)
    else:
        training_set.append(s)

training_sample_set = SampleSet(training_set, iris.feature_names, iris.target_names)
testing_sample_set = SampleSet(test_set, iris.feature_names, iris.target_names)

In [ ]:
dt = DecisionTree.learn_tree(training_sample_set)

The ``__repr__()`` method on the DecisionTree class is used when we call print.
Without this method the next cell would output something like `<__main__.DecisionTree object at 0x1271e3d60>`, with this method defined we can print an object and get something much more informative.

In [ ]:
print(dt)

In [ ]:
# use the decision tree to classify the test set
dt.classify(testing_sample_set)